# Crop Yield Prediction (W18)

This week we will work with the [Crop Yield Prediction dataset](https://www.kaggle.com/datasets/patelris/crop-yield-prediction-dataset) from Kaggle. The `yield_df.csv` file contains roughly 28,000 rows summarizing annual crop yields across countries from 1990 onward. Each row corresponds to one (country, crop, year) combination, with a few weather and management features attached. The yield itself is given in hectograms per hectare (`hg/ha_yield`, where 1 hg/ha = 0.1 kg/ha).

For a bit of real-world context: yields and pesticide use are pulled from the [FAO](https://www.fao.org/faostat/en/#data), while rainfall and temperature come from the [World Bank](https://data.worldbank.org/). Predicting crop yields is a sub-field of its own (see this [survey](https://www.sciencedirect.com/science/article/pii/S0168169920302301), or projects such as [CropNet](https://github.com/fudong03/CropNet) and [CropProphet](https://www.cropprophet.com/systematic-grain-trading/)). Farmers, traders, and governments all use such predictions to make decisions about planting, storage, futures positions, and food security.

The point of this week's workshop is that, in those decision settings, a single number is rarely enough. A forecast of "50 hg/ha ± 5" leads to very different decisions than "50 ± 30". So the focus here is **uncertainty quantification**: not just predicting yields, but understanding the *distribution* of plausible outcomes around those predictions.

A few notes on the columns:

- `Area` is the country.
- `Item` is the crop type (Maize, Wheat, Potatoes, Rice paddy, Soybeans, Sorghum, ...).
- `Year` is the year of observation (1990 onward).
- `hg/ha_yield` is the **target**, given in hectograms per hectare.
- `average_rain_fall_mm_per_year` is the annual rainfall in mm.
- `pesticides_tonnes` is pesticide use in tonnes.
- `avg_temp` is the average temperature in °C.

Don't be afraid to ask if anything in the data is unclear!

Below you'll find some possible starting points. Pick the level that best suits you, dig in, or ignore them and do your own thing. Happy coding!

##### **Beginner**
- Start by getting oriented with `.head()`, `.info()`, and `.describe()`. How many rows are there, and how many distinct countries and crops? Use [`value_counts`](https://pandas.pydata.org/docs/reference/api/pandas.Series.value_counts.html) and [`groupby`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html). Are some crops or countries much more represented than others?
- Pick a single crop (for example, Maize) and plot the distribution of `hg/ha_yield` across all (country, year) observations using [`hist`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.hist.html). Try a few different bin counts. What does the shape look like: symmetric, skewed, multi-modal?
- Overlay histograms for two or three different crops on the same axes, or use small multiples. Are the shapes similar across crops, or do they look quite different?
- A common assumption when starting out is that a continuous variable is approximately [normally distributed](https://en.wikipedia.org/wiki/Normal_distribution). Looking at the histograms above, would you say a Gaussian is a reasonable assumption for `hg/ha_yield`? Why or why not?
- How do yields evolve over time? Pick a country, group by `Year`, and plot the average `hg/ha_yield` per year using [`groupby`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html) and [`plot`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.plot.html). Are there visible trends?
- Make scatter plots of `hg/ha_yield` against the weather features (`average_rain_fall_mm_per_year`, `avg_temp`) and against `pesticides_tonnes`. Do you see any obvious relationships? You may want to filter to a single crop at a time so the picture is interpretable.

##### **Intermediate**
The Beginner section asked whether yields *look* Gaussian. Now let's quantify that. A common approach to uncertainty quantification is to first fit a distribution to the data, and then check how well it describes the tails (which is where the costly events tend to live). If you skipped the Beginner section, it's a good idea to first plot a few histograms so you know what shape the data has.

- Pick a (country, crop) combination with a reasonable number of observations (for example, United States and Maize, or India and Rice paddy) and filter the dataframe down to those rows.
- Fit a Gaussian to the yields by [maximum likelihood estimation](https://en.wikipedia.org/wiki/Maximum_likelihood_estimation) (MLE). For a Gaussian, the MLE has a closed form: it is just the sample mean and sample variance. You can compute these directly with `x.mean()` and `x.std(ddof=0)`, or use [`scipy.stats.norm.fit`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.norm.html), which returns `(loc, scale)`. Plot the histogram and overlay the fitted PDF.
- A picture of the central mass can be misleading. What often matters more for risk is the tails. Make a [QQ plot](https://en.wikipedia.org/wiki/Q%E2%80%93Q_plot) of the data against the fitted Gaussian using [`scipy.stats.probplot`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.probplot.html) or [`statsmodels.api.qqplot`](https://www.statsmodels.org/stable/generated/statsmodels.graphics.gofplots.qqplot.html). Where do the points lie above or below the diagonal? What does that tell you about the empirical tails compared to a Gaussian's?
- Try a heavier-tailed alternative such as the [Student-t](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.t.html) distribution, or a skewed alternative such as the [log-normal](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.lognorm.html). Fit each by MLE using its `.fit()` method, and produce the corresponding QQ plot. Which family captures the tails best for your chosen series?
- Repeat for two or three other (country, crop) combinations. Does the same family fit well across all of them, or does the right choice depend on the crop or country?

##### **Advanced**
We now move from describing a single distribution to *predictive* uncertainty: we want a distribution of plausible yields conditional on inputs like country, year, and weather. If you start here, first make sure you understand the dataset through some quick exploratory analysis.

- Build a baseline point predictor. Pick features (for example, one-hot encoded `Area` and `Item`, plus `Year`, rainfall, pesticides, and temperature) and fit either a [`LinearRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html) or a [`HistGradientBoostingRegressor`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.HistGradientBoostingRegressor.html) to predict `hg/ha_yield`. Use a [`train_test_split`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) and report mean absolute error and RMSE on the held-out set. This is your point-prediction baseline.
- Now move from point predictions to **quantile predictions**. Instead of asking "what is the expected yield?", we ask "what is the 10th, 50th, and 90th percentile yield?". The standard tool here is [quantile regression](https://en.wikipedia.org/wiki/Quantile_regression), which replaces the squared-error loss with the [pinball loss](https://scikit-learn.org/stable/modules/model_evaluation.html#pinball-loss). Try [`QuantileRegressor`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.QuantileRegressor.html) (linear) or [`HistGradientBoostingRegressor(loss="quantile", quantile=tau)`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.HistGradientBoostingRegressor.html). Fit one regressor per quantile (say 0.1, 0.5, 0.9) and produce a prediction interval `[q10, q90]` for every test row. The [scikit-learn quantile gradient boosting example](https://scikit-learn.org/stable/auto_examples/ensemble/plot_gradient_boosting_quantile.html) is a useful reference.
- How good are those intervals? What fraction of test observations actually fall inside their predicted `[q10, q90]` interval? If your model is well calibrated, this should be close to 80%. Compute the **empirical coverage** and the **average interval width**, and compare across crops. Is it possible to have high coverage with narrow intervals, or do they trade off?
- Tie it back to a decision: if you were a trader holding a long position in a crop futures contract, or a government building a strategic grain reserve, how would you actually use the predicted distribution of yields? You don't need to implement this end-to-end, but argue from the intervals or quantiles you produced.

### Helpful references
- [Maximum likelihood estimation (Wikipedia)](https://en.wikipedia.org/wiki/Maximum_likelihood_estimation)
- [`scipy.stats` distribution objects and `.fit()`](https://docs.scipy.org/doc/scipy/reference/stats.html)
- [statsmodels QQ plot reference](https://www.statsmodels.org/stable/generated/statsmodels.graphics.gofplots.qqplot.html)
- [Scikit-learn quantile regression user guide](https://scikit-learn.org/stable/modules/linear_model.html#quantile-regression)
- [Prediction intervals for gradient boosting regression (sklearn example)](https://scikit-learn.org/stable/auto_examples/ensemble/plot_gradient_boosting_quantile.html)
- [Crop yield prediction survey (Computers and Electronics in Agriculture, 2020)](https://www.sciencedirect.com/science/article/pii/S0168169920302301)

## Code

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("yield_df.csv", index_col=0)
df.head()

,Area,Item,Year,hg/ha_yield,average_rain_fall_mm_per_year,pesticides_tonnes,avg_temp
0,Albania,Maize,1990,36613,1485.0,121.0,16.37
1,Albania,Potatoes,1990,66667,1485.0,121.0,16.37
2,Albania,"Rice, paddy",1990,23333,1485.0,121.0,16.37
3,Albania,Sorghum,1990,12500,1485.0,121.0,16.37
4,Albania,Soybeans,1990,7000,1485.0,121.0,16.37
